# 01 - Data Understanding

## Objetivo

Esta etapa tem como objetivo compreender a estrutura do conjunto de dados antes do início das etapas análise exploratória, limpeza e modelagem.

Serão investigadas características como:

- Dimensionalidade do dataset;
- Tipos das variáveis;
- Valores ausentes;
- Distribuições iniciais;
- Estatísticas descritivas;
- Possíveis inconsistências;
- Definição da variável alvo.

Ao final desta etapa será possível compreender a estrutura geral dos dados e identificar os primeiros desafios para a construção do modelo de predição:

- Quantas observações existem?

- Quantas variáveis existem?

- Quais são numéricas?

- Quais são categóricas?

- Existem valores ausentes?

- Existem variáveis redundantes?

- Existe alguma variável que não deverá entrar no modelo?

- Qual variável será utilizada como alvo?

In [77]:
from pathlib import Path
import pandas as pd

# Caminho do projeto
PROJECT_ROOT = Path.cwd().parent # cwd: current working directory
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Carregamento dos dados
csv_file = DATA_PATH / "data.csv"
df = pd.read_csv(csv_file)

In [78]:
# Dimensão do dataset
n_samples, n_features = df.shape

print("=" * 35)
print("DIMENSÃO DO DATASET")
print("=" * 35)
print(f"Número de observações : {n_samples}")
print(f"Número de atributos   : {n_features}")

DIMENSÃO DO DATASET
Número de observações : 20805
Número de atributos   : 9


In [79]:
df.head()

,produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
0,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/14/2017,"20,00","30,00",ANTENA
1,23,CABO RG06,NaN,15.0,NaN,2/14/2017,NaN,NaN,CABO
2,40,CONECTOR ANILHA RG06,"1,00",1.0,"1,00",2/14/2017,NaN,NaN,CABO
3,38,CONECTOR PRESSAO RG06,NaN,1.0,NaN,2/14/2017,NaN,NaN,CABO
4,65,CONVERSOR DIGITAL BEDIN,"110,00",1.0,"110,00",2/14/2017,"85,00","25,00",CONVERSOR


## Estrutura do conjunto de dados

### Valores faltantes e tipos de variáveis

In [80]:
print("Informações sobre o dataset:")
df.info()

Informações sobre o dataset:
<class 'pandas.DataFrame'>
RangeIndex: 20805 entries, 0 to 20804
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   produto         20805 non-null  int64  
 1   descricao       20805 non-null  str    
 2   valor_unitario  18923 non-null  str    
 3   quantidade      20425 non-null  float64
 4   valor_total     19473 non-null  str    
 5   data_venda      20799 non-null  str    
 6   preco_custo     8676 non-null   str    
 7   saldo           9009 non-null   str    
 8   grupo           20805 non-null  str    
dtypes: float64(1), int64(1), str(7)
memory usage: 1.4 MB


A análise da estrutura do banco de dados inicial indica que o conjunto é composto por 20.805 observações distribuídas em nove atributos. Entre os atributos disponíveis, temos as seguintes informações:

| Variável     | Tipo (Pandas) | Categoria         | Descrição                                                          | Unidade    |
| ------------ | ------------- | ----------------- | ------------------------------------------------------------------ | ---------- |
| **produto**  | `int64`         | Numérica discreta        | ID do produto.                                                | quantidade          |
| **descricao** | `str`         | Categórica        | Descrição do produto.                              | —          |
| **valor_unitario**     | `float64`       | Numérica | Valor por unidade.                                                    | R$         |
| **quantidade** | `int64`       | Numérica discreta | Quantidade vendida.                                   | quantidade |
| **valor_total**   | `float64`       | Numérica | Valor total da venda.                            | R$ |
| **data_venda**     | `datetime`         | Data        | Data da venda.                  | Data          |
| **preco_custo**     | `float64`       | Numérica | Valor de custo do produto.                                           | R$         |
| **saldo**    | `float64`       | Numérica | Lucro sob a venda. | R$         |
| **grupo**    | `str`       | Categórica | Grupo de venda. | —         |

Nota-se que as variáveis `valor_unitario`, `valor_total`, `data_venda`, `preco_custo` e `saldo` estão com o tipo errado de dado, onde o Pandas interpretou como `str` quando na verdade deveria ser consideradas variáveis numéricas.

Também temos informações de que:
$$valor\, total = valor\,unitario * quantidade$$
e
$$saldo = valor\, total - preco\, custo$$

Adicionalmente, existem muitos valores faltantes que precisarão de tratamento. Existe a possibilidade de completar `valor_total`,  `quantidade` e `valor_unitario` por exister uma relação entre elas. Porém, deve-se tomar cuidado com a existência de `quantidades` quebradas para produtos que só aceitam inteiros. Além disso, não é possível controlar possíveis **descontos** por venda, pois isso traria um peso maior para o modelo de predição. 

O conjunto de dados também apresenta valores de datas ausentes, devem ser retirados, já que iremos tratar de séries temporais. Também tem-se mais da metadde dos dados de `preco_custo` nulos (~58.30%) e de `saldo` nulos (~56.70%), o que mostra que estes atributos não servirão como *features* para o modelo.

In [81]:
# Valores ausentes
print("Valores ausentes por coluna:")
print(df.isna().sum())

Valores ausentes por coluna:
produto               0
descricao             0
valor_unitario     1882
quantidade          380
valor_total        1332
data_venda            6
preco_custo       12129
saldo             11796
grupo                 0
dtype: int64


In [82]:
print("Número de linhas duplicadas:")
print(df.duplicated().sum())

Número de linhas duplicadas:
1816


In [83]:
df[df.duplicated()]

,produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
5,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/14/2017,"20,00","30,00",ANTENA
6,65,CONVERSOR DIGITAL BEDIN,"110,00",1.0,"110,00",2/14/2017,"85,00","25,00",CONVERSOR
11,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/14/2017,"20,00","30,00",ANTENA
15,12,CONVERSOR DIGITAL VISIONTEC,"110,00",1.0,"110,00",2/15/2017,"75,00","35,00",CONVERSOR
17,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/15/2017,"20,00","30,00",ANTENA
...,...,...,...,...,...,...,...,...,...
20722,278,ANTENA UHF 34 ELEMENTOS,"70,00",1.0,"70,00",8/4/2025,"30,00","40,00",ANTENA
20764,278,ANTENA UHF 34 ELEMENTOS,"70,00",1.0,"70,00",8/13/2025,"30,00","40,00",ANTENA
20767,188,CABO COAXIAL RG 59,"2,50",10.0,"22,00",8/13/2025,NaN,"9,00",CABO
20773,43,CONECTOR ROSCA,"1,00",2.0,"2,00",8/13/2025,NaN,NaN,CABO


Esses valores não são duplicatas reais, pois são vendas existentes e com diferentes IDs de pedidos.

## Hipóteses Iniciais

Com base no entendimento do negócio em consonância com o conjunto de dados, a compreensão inicial das variáveis ajuda a levantar algumas hipóteses antes da realização do Pré-processamento e da Análise Exploratória de Dados (EDA). Essas hipóteses servirão como ponto de partida para orientar as próximas etapas do projeto e deverão ser confirmadas ou refutadas por meio de análises estatísticas e visualizações.

### Hipótese 1 - ID do Produto

A variável `produto` apenas identifica o produto vendido. De primeiro momento, considerando que é uma variável com alta cardinalidade, não se mostra relevante no sentido de calcular o faturamento total mensal **por produto**, pois isso atrapalharia o modelo. Entretanto, se houver tempo hábil, seria interessante investigar os produtos com maiores faturamentos mensais separadamente e, para isso, é necessário utilizar o identificador do produto.

---

### Hipótese 2 - Descrição do Produto

Essa variável só servirá para entender o negócio como um todo e para análises específicas. Ela será interessante para entender possíveis *outliers* que possam aparecer na etapa de EDA.

---

### Hipótese 3 - Quantidade

Não é permitido ter valores quebrados para esta variável, exceto se houver o entendimento do negócio para produtos específicos.

---

### Hipótese 4 - Grupo

Essa variável será interessante para entender quais grupos contribuem mais para o faturamento global da empresa mensalmente e entender se há grupos com alta cardinalidade que podem ser removidor sem prejudicar a análise.

---

## Pré-processamento dos dados

### Etapa para tratamento de nome e tipo de variáveis.

In [84]:
banco_inicial = df.copy()

banco_inicial.rename(columns={'produto': 'id_produto'}, inplace=True)

In [85]:
# Modificando as colunas de valores monetários para o formato numérico

colunas_monetarias = [
    'valor_unitario',
    'valor_total',
    'preco_custo',
    'saldo'
]

for col in colunas_monetarias:
    banco_inicial[col] = (
        banco_inicial[col]
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    banco_inicial[col] = pd.to_numeric(
            banco_inicial[col],
            errors='coerce' # Converte valores não numéricos para NaN
        )

In [86]:
# Modificando a coluna de data para o formato datetime
banco_inicial['data_venda'] = pd.to_datetime(
    banco_inicial['data_venda'],
    errors='coerce' # Converte valores não convertíveis para NaT
)

In [87]:
banco_inicial.info()

<class 'pandas.DataFrame'>
RangeIndex: 20805 entries, 0 to 20804
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      20805 non-null  int64         
 1   descricao       20805 non-null  str           
 2   valor_unitario  18923 non-null  float64       
 3   quantidade      20425 non-null  float64       
 4   valor_total     19473 non-null  float64       
 5   data_venda      20799 non-null  datetime64[us]
 6   preco_custo     8676 non-null   float64       
 7   saldo           9009 non-null   float64       
 8   grupo           20805 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.4 MB


### Tratamento de valores nulos e faltantes

In [88]:
banco_inicial[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

valor_unitario     1882
quantidade          380
valor_total        1332
preco_custo       12129
saldo             11796
dtype: int64

#### Análise de valores faltantes entre `saldo` e `preco_custo`

In [89]:
banco_inicial.groupby('grupo')[['preco_custo', 'saldo']].apply(
    lambda x: x.isna().sum()
)

,preco_custo,saldo
grupo,,
ANTENA,1420,1445
CABO,4773,4260
COMPONENTES ANTENAS,2119,2160
CONVERSOR,265,267
INTERFONE,321,323
PORTAO,1081,1074
PRODUTO ESPECIAL,3,1
RECEPTOR,13,16
REDES,2,2


* Nota-se que, muitos valores faltantes em `preco_custo` e `saldo` impossibilitam a análise de lucros de maneira eficiente, portanto, aqui é uma ótima justificativa para descartar a ideia de **predição de lucro** e focar apenas em *análises de sazonalidade*, *predições de demanda* e *faturamento mensal*

#### Analisando valores faltantes de `valor_total` e `valor_unitario`

* Como a variável *valor_total* é dada por *valor_total = valor_unitario * quantidade*, é possível preencher alguns valores vazios, da mesma forma aplicarmos que *valor_unitario = valor_total / quantidade*

In [90]:
# Preenchendo valores ausentes de valor_total e valor_unitario
mask_total = (
    banco_inicial['valor_total'].isna()
    & banco_inicial['valor_unitario'].notna()
    & banco_inicial['quantidade'].notna()
)

banco_inicial.loc[mask_total, 'valor_total'] = (
    banco_inicial.loc[mask_total, 'valor_unitario']
    * banco_inicial.loc[mask_total, 'quantidade']
)

mask_unitario = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['valor_total'].notna()
    & banco_inicial['quantidade'].notna()
    & (banco_inicial['quantidade'] > 0)
)

banco_inicial.loc[mask_unitario, 'valor_unitario'] = (
    banco_inicial.loc[mask_unitario, 'valor_total']
    / banco_inicial.loc[mask_unitario, 'quantidade']
)

In [91]:
# Verificando novamente os valores ausentes após o preenchimento
banco_inicial[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

valor_unitario      794
quantidade          380
valor_total         906
preco_custo       12129
saldo             11796
dtype: int64

* Reduziu-se o número de valor faltandes de valor_unitario de 1882 para 794 (melhora de 57.81%) e o número de valores faltantes de valor_total de 1332 para 906 (melhora de 31.98%).

* Por fim, retira-se os registros com `valor_total`, `valor_unitario` e `quantidade` nulos ou ausentes.

In [92]:
banco_inicial = banco_inicial.dropna(
    subset=['valor_unitario', 'quantidade', 'valor_total'],
    how='all'
)

In [93]:
# Número de registros restantes no banco de dados
print(f"O número de registros restantes no banco de dados é: {banco_inicial.shape[0]}")

O número de registros restantes no banco de dados é: 20802


#### Registros com `valor_total` preenchido, mas `valor_unitario` e `quantidade` ausentes

In [94]:
mask = (
    banco_inicial['valor_total'].notna()
    & banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].isna()
)

mask.sum()

np.int64(9)

* Sem as variáveis `quantidade` e `valor_unitario` é difícil de realizar boas análises, então é interessante excluir estes registros, considerando que são poucos e não afetarão o modelo final.

In [95]:
banco_inicial = banco_inicial[~mask].copy()

#### Registros com `quantidade` vazia, mas `valor_unitario` e `valor_total` preenchidos

In [96]:
mask_qtdna = (
    banco_inicial['valor_unitario'].notna()
    & banco_inicial['quantidade'].isna()
    & banco_inicial['valor_total'].notna()
)

mask_qtdna.sum()

np.int64(247)

In [97]:
# Calculando a quantidade a partir do valor_total e valor_unitario
qtd_calc = (
    banco_inicial.loc[mask_qtdna, 'valor_total']
    / banco_inicial.loc[mask_qtdna, 'valor_unitario']
)
print(qtd_calc)

410      2.5
414      2.5
2463     1.3
3285     1.5
7226     0.5
        ... 
20703    0.5
20715    0.5
20719    3.5
20740    0.5
20768    0.5
Length: 247, dtype: float64


In [98]:
# Verificando quantos valores calculados são inteiros
print((qtd_calc % 1 == 0).sum())

2


In [99]:
banco_inicial = banco_inicial[~mask_qtdna].copy()

* A quantidade calculada mostra valores quebrados, mas isso não é coerente com a quantidade vendida de produtos nesta loja. O que mostra que necessariamente o valor cobrado no final da compra (`valor_total`) foi diferente de `valor_unitario` * `quantidade`, o que mostra possíveis descontos ou reajustes no ato da venda.

* Como construir uma variável de reajuste para poucos dados (cerca de 1.2%) introduziria ruídos nas predições, é válido retirar do banco de dados.

#### Registros com `valor_unitario` vazia, mas `quantidade` e `valor_total` preenchidos

In [100]:
mask_vuna = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].notna()
    & banco_inicial['valor_total'].notna()
)

mask_vuna.sum()

np.int64(0)

#### Registros com `valor_unitario`, `quantidade` e `valor_total` vazios

In [101]:
mask_numna = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].isna()
    & banco_inicial['valor_total'].isna()
)

mask_numna.sum()

np.int64(0)

#### Análise de Datas

In [102]:
# Número de registros restantes no banco de dados
print(f"O número de registros restantes no banco de dados é: {banco_inicial.shape[0]}")

O número de registros restantes no banco de dados é: 20546


In [103]:
banco_inicial.info()

<class 'pandas.DataFrame'>
Index: 20546 entries, 0 to 20804
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      20546 non-null  int64         
 1   descricao       20546 non-null  str           
 2   valor_unitario  19764 non-null  float64       
 3   quantidade      20425 non-null  float64       
 4   valor_total     19643 non-null  float64       
 5   data_venda      20540 non-null  datetime64[us]
 6   preco_custo     8636 non-null   float64       
 7   saldo           8995 non-null   float64       
 8   grupo           20546 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.6 MB


Há `valor_unitario`, `quantidade`, `valor_total` e `data_venda` nulos, os quais não serão tratados devido às limitações do dataset disponível. Logo, devem ser retirados. 

In [105]:
# Número de registros com data de venda faltantes
mask_datna = (
    banco_inicial['data_venda'].isna()
)

mask_datna.sum()

np.int64(6)

In [106]:
# Retirando os registros com datas faltantes
banco_inicial = banco_inicial[~mask_datna].copy()

#### Quantidades nulas ou negativas

In [107]:
# Número de registros com quantidade menor ou igual a zero
banco_inicial[banco_inicial['quantidade'] <= 0].shape[0]

0

In [108]:
# Número de registros com valores_unitarios menor ou igual a zero
banco_inicial[banco_inicial['valor_unitario'] <= 0].shape[0]

3

In [109]:
banco_inicial[banco_inicial['valor_unitario'] <= 0]

,id_produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
3658,42,EMENDA,0.0,3.0,0.0,2018-04-23,NaN,NaN,CABO
5877,101,SUP CAVALETE 1,0.0,1.0,0.0,2019-03-22,NaN,NaN,COMPONENTES ANTENAS
7871,0,FRENTE AGL 16 PONTOS,0.0,1.0,0.0,2020-07-29,0.0,0.0,PRODUTO ESPECIAL


In [110]:
# Número de registros com valor_total menor ou igual a zero
banco_inicial[banco_inicial['valor_total'] <= 0].shape[0]

3

In [111]:
banco_inicial[banco_inicial['valor_total'] <= 0]

,id_produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
3658,42,EMENDA,0.0,3.0,0.0,2018-04-23,NaN,NaN,CABO
5877,101,SUP CAVALETE 1,0.0,1.0,0.0,2019-03-22,NaN,NaN,COMPONENTES ANTENAS
7871,0,FRENTE AGL 16 PONTOS,0.0,1.0,0.0,2020-07-29,0.0,0.0,PRODUTO ESPECIAL


In [112]:
# Retirando as inconsistências de quantidade, valor_unitario e valor_total
banco_inicial.drop([3658, 5877, 7871], inplace=True)

In [113]:
banco_inicial.reset_index(drop=True, inplace=True)

In [114]:
banco_inicial.info()

<class 'pandas.DataFrame'>
RangeIndex: 20537 entries, 0 to 20536
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      20537 non-null  int64         
 1   descricao       20537 non-null  str           
 2   valor_unitario  19755 non-null  float64       
 3   quantidade      20416 non-null  float64       
 4   valor_total     19634 non-null  float64       
 5   data_venda      20537 non-null  datetime64[us]
 6   preco_custo     8631 non-null   float64       
 7   saldo           8990 non-null   float64       
 8   grupo           20537 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.4 MB


In [115]:
# Retirando os registros com valores ausentes em quantidade, valor_unitario e valor_total
banco_inicial.dropna(subset=['valor_unitario', 'quantidade', 'valor_total'], inplace=True)

In [116]:
banco_inicial.info()

<class 'pandas.DataFrame'>
Index: 19634 entries, 0 to 20536
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      19634 non-null  int64         
 1   descricao       19634 non-null  str           
 2   valor_unitario  19634 non-null  float64       
 3   quantidade      19634 non-null  float64       
 4   valor_total     19634 non-null  float64       
 5   data_venda      19634 non-null  datetime64[us]
 6   preco_custo     8458 non-null   float64       
 7   saldo           8932 non-null   float64       
 8   grupo           19634 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.5 MB


In [117]:
# Número de registros restantes no banco de dados
print(f"O número de registros restantes no banco de dados é: {banco_inicial.shape[0]}")

O número de registros restantes no banco de dados é: 19634


In [126]:
banco_inicial.to_csv(
    "../data/interm/data_1.csv",
    index=False
)

---

## Primeiras Análises Estatísticas

In [118]:
banco_inicial.describe()

,id_produto,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo
count,19634.000000,19634.000000,19634.000000,19634.000000,19634,8458.000000,8932.000000
mean,218.267342,72.463404,4.183559,96.755103,2021-04-28 20:04:12.297035,69.029558,46.740596
min,0.000000,0.500000,1.000000,1.000000,2017-02-14 00:00:00,0.000000,-14179.000000
25%,44.000000,5.000000,1.000000,16.000000,2018-12-03 00:00:00,15.000000,13.000000
50%,233.000000,35.000000,1.000000,44.000000,2021-05-31 00:00:00,25.000000,25.000000
75%,318.000000,70.000000,2.000000,90.000000,2023-05-31 00:00:00,70.000000,40.000000
max,737.000000,7536.000000,400.000000,9720.000000,2025-10-30 00:00:00,17160.000000,7560.000000
std,167.363054,172.379914,12.511824,244.465625,NaN,291.181147,242.768560


### Interpretação das Estatísticas Descritivas

As variáveis `preco_custo` e `saldo` apresentam muitos registros com valores faltantes e inconsistências, como valores nulos e negativos. Estas variáveis serão descartadas do modelo.

A variável `valor_unitario` possui uma mediana (R$ 35) abaixo da média (R$ 72.46), o que mostra uma distribuição positiva, isto é, assimétrica a direita. Também é possível comprovar isto observando o desvio padrão e o valor máximo da variável, o que deixa evidente a existência de possíveis *outliers*.


De maneira similar, a variável `valor_total` também possui uma assimetria direita, já que ela é construída a partir da variável `valor_unitario`.

---

# Conclusão

Apesar de ser uma boa base de dados para se criar modelos de predição, é necessário alguns tratamentos. 

1) Todos os grupos devem ser analisados no faturamento?

2) A alta cardinalidade de diferentes produtos irá afetar a modelagem? Se sim, compensa pensar em faturamento por grupo e não por produtos individualmente.

3) O faturamento é justamente a variável `valor_total`, logo ela é construída pelas variáveis `quantidade` e `valor_unitario`, isto significa que está ocorrendo **data leakage**?

4) Quais normalizações devem ser feitas nas variáveis numéricas para tratamento de valores extremos?